In [82]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import csv

from model.dataset import BagsDataset, custom_collate_fn

In [83]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [84]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Visium/HumanOvarianCancer/T_cell.h5ad')

In [85]:
dataset = BagsDataset(adata, immune_cell='tcell', radius=150, n_genes=500, resolution='low')
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
#dataset = load_datasets(data_dir='/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE', immune_cell='tcell',radius=200,)

Tumor cells shape after filtering: (1226, 17943)
Selecting top 500 genes based on mean expression
Percentile value: 4.037618160247803
adata.obs[T] after binarization: AAACAAGTATCTCCCA-1    0
AAACAATCTACTAGCA-1    0
AAACACCAATAACTGC-1    1
AAACAGAGCGACTCCT-1    1
AAACAGCTTTCAGAAG-1    0
Name: T, dtype: int64
Preprocessed data: (3455, 500)


Creating Bags with radius 150: 100%|█████████████████████████| 3455/3455 [00:00<00:00, 22858.34it/s]

Total bags created: 1226
Average instances per bag: 4


In [42]:
dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)



In [43]:
distances, gene_expressions, labels, core_index, current_genes = next(iter(dataloader))


In [63]:
class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(-3.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=0)
        self.softmax = nn.Softmax(dim=0)
    
    def forward(self, x):
        #print(x)
        a = self.a
        x = self.softmax(-torch.exp(a) * x)
        return x

In [64]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[8.7300e-04],
        [8.7300e-04],
        [8.7300e-04],
        [9.9738e-01]], grad_fn=<SoftmaxBackward0>)


In [65]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        b = self.b
        x = self.softmax(torch.exp(b) * x)
        return x

In [66]:
gene_expressions[0]

tensor([[4.8622, 4.3626, 4.3836,  ..., 0.6681, 1.6173, 1.6173],
        [5.4120, 4.6098, 5.0564,  ..., 1.3507, 1.0670, 1.1708],
        [5.8429, 5.3950, 5.2942,  ..., 1.1777, 1.6183, 1.4221],
        [5.3259, 4.6904, 4.3758,  ..., 1.5341, 1.3158, 1.3940]])

In [67]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output)

tensor([[3.1576e-01, 8.1198e-02, 8.5963e-02,  ..., 3.5318e-06, 4.6623e-05,
         4.6623e-05],
        [4.7274e-01, 5.3407e-02, 1.7983e-01,  ..., 7.5870e-06, 3.5091e-06,
         4.6525e-06],
        [5.6870e-01, 1.6831e-01, 1.2797e-01,  ..., 1.7678e-06, 5.8556e-06,
         3.4350e-06],
        [5.5433e-01, 9.8542e-02, 4.1893e-02,  ..., 1.8511e-05, 1.0228e-05,
         1.2649e-05]], grad_fn=<SoftmaxBackward0>)


In [68]:
distances, gene_expressions, labels, core_index, current_genes = next(iter(dataloader))

In [69]:
class Immunogenicity(nn.Module):
    def __init__(self, all_genes):
        super(Immunogenicity, self).__init__()
        self.all_genes = all_genes
        self.gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}
        self.ig = nn.Parameter(torch.full((len(all_genes),), -1.0), requires_grad=True)
    
    def forward(self, current_genes):
        # Filter genes to include only those present in all_genes
        filtered_genes = [gene for gene in current_genes if gene in self.gene_to_index]
        indices = [self.gene_to_index[gene] for gene in filtered_genes]
        ig = torch.sigmoid(self.ig[indices])
        return ig, filtered_genes

In [70]:
all_genes = pd.read_csv('./data/tumor_antigens.csv')
all_genes = all_genes['Gene'].values.tolist()

In [71]:
model = Immunogenicity(all_genes)

In [72]:
output, filtered_genes = model(list(current_genes[0]))

In [73]:
len(output)

10

In [74]:
class MIL(nn.Module):
    def __init__(self, all_genes):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.immunogenicity = Immunogenicity(all_genes)
        self.alpha = nn.Parameter(torch.tensor(1.0), requires_grad=True)
        self.beta = nn.Parameter(torch.tensor(1.0), requires_grad=True)
    
    def forward(self, distances, gene_expressions, current_genes):
        bag_outputs = []
        for distance, gene_expression in zip(distances, gene_expressions):
            distance = self.distance(distance)
            gene_expression = self.gene_expression(gene_expression)
            immunogenicity, filtered_genes = self.immunogenicity(current_genes)
            alpha = self.alpha
            beta = self.beta
        
            if len(filtered_genes) == 0:
                continue  # Skip if no overlapping genes
        
        # Print debug information
            #print(f"gene_expression shape: {gene_expression.shape}")
            #print(f"current_genes length: {len(current_genes)}")
            #print(f"filtered_genes length: {len(filtered_genes)}")
        
        
            gene_to_index = {gene: idx for idx, gene in enumerate(current_genes)}
        
            gene_indices = [gene_to_index[gene] for gene in filtered_genes if gene in gene_to_index]
            gene_expression = gene_expression[:, gene_indices]
        
            #print(f"Filtered gene_expression shape: {gene_expression.shape}")
            #print(f"Immunogenicity shape: {immunogenicity.shape}")
        
            z = gene_expression @ immunogenicity
            #print(f"z shape: {z.shape}")
            z = z.unsqueeze(1)
            #print(f"z shape: {z.shape}")
            #print(f"distance shape: {distance}")
            bag_output = distance * z
            bag_output = torch.sum(bag_output, dim=0)
            bag_output = torch.exp(alpha) * bag_output + beta
            bag_output = torch.sigmoid(bag_output)
            #print(bag_output)
            bag_outputs.append(bag_output)
            #df = pd.DataFrame(bag_outputs)
            #df.to_csv('output.csv')
    
        
        return torch.stack(bag_outputs).squeeze(dim=1)

In [75]:
model = MIL(all_genes)

In [76]:
output = model(distances, gene_expressions, list(current_genes[0]))
output

tensor([0.7311], grad_fn=<SqueezeBackward1>)

In [77]:
labels[0]


tensor(0.)

In [78]:
ig_scores_before_training = model.immunogenicity.ig

In [79]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = np.inf
        self.counter = 0
        self.stopped_epoch = 0
        self.early_stop = False

    def __call__(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), 'final_model.pth')
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch

In [80]:

model = MIL(all_genes).to(device)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
num_epochs = 5
patience = 5
early_stopping = EarlyStopping(patience=patience, delta=0.1)

In [81]:
for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, current_genes) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()

            distances = torch.stack(distances).to(device)
            gene_expressions = torch.stack(gene_expressions).to(device)
            label = label.clone().detach().float().to(device)
            
            output = model(distances, gene_expressions, list(current_genes[0]))
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    

    model.eval()
    val_loss = 0.0
    val_predictions = []
    val_labels = []
    with torch.no_grad():
        for val_distances, val_gene_expressions, val_label, _, val_current_genes in val_dataloader:
            val_distances = torch.stack(val_distances).to(device)
            val_gene_expressions = torch.stack(val_gene_expressions).to(device)
            val_label = val_label.clone().detach().float().to(device)
            val_output = model(val_distances, val_gene_expressions, list(val_current_genes[0]))
            val_loss += criterion(val_output, val_label).item()
            val_predictions.extend(val_output.cpu().numpy())
            val_labels.extend(val_label.cpu().numpy())
    
    val_loss /= len(val_dataloader)
    val_auroc = roc_auc_score(val_labels, val_predictions)
    print(f'Validation Loss: {val_loss:.4f}, Validation AUROC: {val_auroc:.4f}')

    """early_stopping(val_loss, model, epoch)
    if early_stopping.early_stop:
        print(f'Early stopping at epoch {epoch+1}')
        break"""
        

Epoch 1/5: 100%|██████████| 858/858 [00:01<00:00, 549.14batch/s, loss=0.951]


Epoch [1/5], Loss: 0.6995
Validation Loss: 0.7165, Validation AUROC: 0.4573


Epoch 2/5:  70%|██████▉   | 600/858 [00:01<00:00, 567.44batch/s, loss=0.837]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

Epoch 3/5: 100%|██████████| 858/858 [00:01<00:00, 565.51batch/s, loss=0.802]


Epoch [3/5], Loss: 0.6952
Validation Loss: 0.6948, Validation AUROC: 0.4572


Epoch 4/5:  81%|████████▏ | 698/858 [00:01<00:00, 563.39batch/s, loss=0.48] IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

Epoch 5/5: 100%|██████████| 858/858 [00:01<00:00, 554.51batch/s, loss=0.823]


Epoch [5/5], Loss: 0.7014
Validation Loss: 0.6991, Validation AUROC: 0.4570


In [35]:
ig_scores_after_training = model.immunogenicity.ig

with open('ig_score_changes.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score Before Training', 'IG Score After Training'])
    for gene, before, after in zip(all_genes, ig_scores_before_training, ig_scores_after_training):
        writer.writerow([gene, before.item(), after.item()])

NameError: name 'csv' is not defined